In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_excel('/content/logs_KX7A_20260212-1051 - Copy.xlsx')


In [ ]:
eventos_ver = [
    'Módulo de curso visto',
    'El estado del envío ha sido visto.',
    'Formato de envío visto.'
]

eventos_responder = [
    'Se ha enviado un envío',
    'Envio creado.'
]

In [ ]:
df_ver = df[df['Nombre del evento'].isin(eventos_ver)]
df_resp = df[df['Nombre del evento'].isin(eventos_responder)]

In [ ]:
df_ver = df_ver[['Nombre completo del usuario','Contexto del evento','Hora']]
df_ver = df_ver.rename(columns={'Hora':'Hora_ver'})

df_resp = df_resp[['Nombre completo del usuario','Contexto del evento','Hora']]
df_resp = df_resp.rename(columns={'Hora':'Hora_resp'})

In [ ]:
df_ver = df_ver[['Nombre completo del usuario','Contexto del evento','Hora']]
df_ver = df_ver.rename(columns={'Hora':'Hora_ver'})

df_resp = df_resp[['Nombre completo del usuario','Contexto del evento','Hora']]
df_resp = df_resp.rename(columns={'Hora':'Hora_resp'})

In [ ]:
lat['Hora_resp'] = pd.to_datetime(lat['Hora_resp'], format='%d/%m/%y, %H:%M:%S')
lat['Hora_ver'] = pd.to_datetime(lat['Hora_ver'], format='%d/%m/%y, %H:%M:%S')
lat['latencia_horas'] = (
    lat['Hora_resp'] - lat['Hora_ver']
).dt.total_seconds() / 3600

In [ ]:
lat = lat[lat['latencia_horas'] >= 0]

In [ ]:
lat_usuario = lat.groupby('Nombre completo del usuario')['latencia_horas'].agg([
    'mean',
    'std',
    'count'
]).reset_index()

lat_usuario.columns = [
    'Usuario',
    'latency_mean',
    'latency_std',
    'response_count'
]

In [ ]:
lat_usuario['latency_cv'] = (
    lat_usuario['latency_std'] /
    lat_usuario['latency_mean']
)

In [ ]:
def clasificar_participacion(mean, cv):

    if mean > 72:        # más de 3 días
        return 2         # Demora prolongada
    elif mean < 1 and cv < 0.25:
        return 2         # Hiper-respuesta
    elif cv < 0.5:
        return 0         # Latencia similar
    else:
        return 1         # Variaciones ocasionales

In [ ]:
lat_usuario['participacion_social'] = lat_usuario.apply(
    lambda x: clasificar_participacion(
        x['latency_mean'],
        x['latency_cv']
    ),
    axis=1
)

In [ ]:
lat_usuario.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

lat_usuario[['mean_norm','cv_norm']] = scaler.fit_transform(
    lat_usuario[['latency_mean','latency_cv']]
)

lat_usuario['riesgo_score'] = (
    lat_usuario['mean_norm'] * 0.6 +
    lat_usuario['cv_norm'] * 0.4
)

In [ ]:
top_10_peor = lat_usuario.sort_values(
    by='riesgo_score',
    ascending=False
).head(10)

top_10_peor

In [ ]:
top_10_mejor = lat_usuario.sort_values(
    by='riesgo_score'
).head(10)

top_10_mejor